# **Decoder Casual Language Model**

In [2]:
import torchtext
import torch
import math
import torch.nn as nn
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import IMDB

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Dataset**

In [3]:
train_iter,val_iter = IMDB()
next(iter(train_iter))

(1,
 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far betwee

In [4]:
# define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cpu'

## Preprocessing Data

This section focuses on preparing text data for NLP tasks, mainly through tokenization and vocabulary creation.

### Special Tokens and Indices
Defines special tokens such as `<unk>`, `<pad>`, and an empty string (used as an EOS token), and assigns them indices (`0`, `1`, and `2`). These tokens serve specific purposes in text processing:

- `UNK_IDX`: Represents unknown or out-of-vocabulary words  
- `PAD_IDX`: Used to pad shorter sequences in a batch for uniform length  
- `EOS_IDX`: Indicates the end of a sequence  

### `yield_tokens` Function
A generator function that iterates through the dataset (`data_iter`), applies a tokenizer to each sample, and yields tokenized outputs one at a time.

### Building the Vocabulary
Constructs a vocabulary from the tokenized data using `build_vocab_from_iterator`. Special tokens are added at the beginning, and all tokens with a minimum frequency of 1 are included.

### Handling Unknown Tokens
Sets a default index (`UNK_IDX`) for tokens not found in the vocabulary, ensuring the model can handle unseen words gracefully.

### `text_to_index` Function
Converts raw text into a sequence of numerical indices based on the vocabulary, making it suitable as input for machine learning models.

### `index_to_en` Function
Transforms a sequence of indices back into readable text, which is useful for interpreting model outputs.

### Functionality Check
Validates the preprocessing pipeline by converting a sample tensor `[0, 1, 2]` back into the corresponding special tokens, ensuring that the mappings are working correctly.

In [5]:
UNK_IDX, PAD_IDX, EOS_IDX = 0, 1, 2
special_tokens = ["<unk>","<pad>", "<eos>" ]

In [6]:
tokenizer = get_tokenizer("basic_english")

def yield_tokens(dataset):
    for id, text in dataset:
        yield tokenizer(text)
        
vocab = build_vocab_from_iterator(yield_tokens(train_iter),specials=special_tokens,
                                  special_first=True)
vocab.set_default_index(UNK_IDX)

In [7]:
vocab["store"]

1024

#### **Build Pipelines**

Text to Indices

In [8]:
def text_pipeline (sample):
    return [vocab[text] for text in tokenizer(sample) ]

text_pipeline("I rented I AM CURIOUS-YELLOW from my video store because of all")

[13, 1153, 13, 230, 24141, 50, 72, 351, 1024, 87, 9, 40]

Indices to Text

In [9]:
vocab.get_itos()[1153]

'rented'

In [10]:
def index_to_en (indices):
    tokens_list = [vocab.get_itos()[index] for index in indices]
    tokens_list = " ".join(tokens_list)
    return tokens_list
    
index_to_en([13, 1153, 13, 230, 24141, 50, 72, 351, 1024, 87, 9, 40])

'i rented i am curious-yellow from my video store because of all'

### Collate Function

For the decoder model, a collate function is designed to process batches of text data. This function takes a block of text as input and returns a transformed version suitable for training.

The transformation is handled by the `get_sample(block_size, text)` function. This function randomly extracts a segment of text (`src_sequence`) along with its shifted counterpart (`tgt_sequence`), which represents the next tokens in the sequence.

It ensures that the sampled text fits within the specified block size. If the original text is shorter than the block size, adjustments are made accordingly. The function ultimately returns both the source and target sequences, which are used as inputs for training the language model.

In [11]:
def get_sample(block_size, text):
    
    text_len = len(text)
    stop_point = 0
    start_point = 0
    seq_differ = text_len-block_size 
    
    if (seq_differ >= 1):
        start_point = torch.randint(low=0, high = seq_differ, size=(1,)).item() 
        stop_point = start_point + block_size
        
        src_sequence = text[start_point:stop_point]
        tgt_sequence = text[start_point+ 1: stop_point+1]
    
    elif (seq_differ <= 0):
        start_point = 0
        stop_point = text_len
        
        src_sequence = text[start_point:stop_point]
        tgt_sequence = text[start_point+1: stop_point]
        
        tgt_sequence.append('<|endoftext|>')
    
    return src_sequence, tgt_sequence
    

In [12]:
batch_size = 1

batch_of_tokens = []
for i in range(batch_size):
    _,text = next(iter(train_iter))
    batch_of_tokens.append(tokenizer(text))
    

In [13]:
text = batch_of_tokens[0][0:100]
text[0:50]

['i',
 'rented',
 'i',
 'am',
 'curious-yellow',
 'from',
 'my',
 'video',
 'store',
 'because',
 'of',
 'all',
 'the',
 'controversy',
 'that',
 'surrounded',
 'it',
 'when',
 'it',
 'was',
 'first',
 'released',
 'in',
 '1967',
 '.',
 'i',
 'also',
 'heard',
 'that',
 'at',
 'first',
 'it',
 'was',
 'seized',
 'by',
 'u',
 '.',
 's',
 '.',
 'customs',
 'if',
 'it',
 'ever',
 'tried',
 'to',
 'enter',
 'this',
 'country',
 ',',
 'therefore']

In [14]:
block_size = 10
src_seq, tgt_seq = get_sample(block_size, text)
src_seq,tgt_seq

(['around',
  'a',
  'young',
  'swedish',
  'drama',
  'student',
  'named',
  'lena',
  'who',
  'wants'],
 ['a',
  'young',
  'swedish',
  'drama',
  'student',
  'named',
  'lena',
  'who',
  'wants',
  'to'])

In [15]:
vocab(src_seq)

[191, 6, 266, 3994, 615, 1272, 831, 6788, 49, 518]

### **Define Collate Function**

In [16]:
from torch.nn.utils.rnn import pad_sequence

In [17]:
block_size = 30
def collate_func (batch):
    src_batch, tgt_batch = [], []
    
    for _,text in batch:
        src_sequence, tgt_sequence = get_sample(block_size, tokenizer(text))
        #define as token indices
        src_sequence = vocab(src_sequence)
        tgt_sequence = vocab(tgt_sequence)
        
        #define as tensors
        src_sequence = torch.tensor(src_sequence,dtype=torch.int64)
        tgt_sequence = torch.tensor(tgt_sequence, dtype=torch.int64)
        
        #append to batch lists
        src_batch.append(src_sequence)
        tgt_batch.append(tgt_sequence)
    
    
    src_batch = pad_sequence(src_batch,batch_first=False, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch,batch_first=False, padding_value=PAD_IDX)
    
    return src_batch.to(device), tgt_batch.to(device)
        

### **Define Dataloaders**

In [18]:
from torch.utils.data import DataLoader

In [19]:
train_dataloader = DataLoader(train_iter,batch_size=batch_size,
                              shuffle=True, collate_fn=collate_func)

val_dataloader = DataLoader(val_iter, batch_size=batch_size,
                            shuffle=True, collate_fn=collate_func)


### Iterating through data samples

The provided code iterates through batches of source-target pairs from a data loader. It demonstrates how to access and print a few samples from the dataset:

- We initialize an iterator over the data loader named `dataset`.
- A loop runs for 10 iterations to fetch and print the first 10 source-target pairs. For each pair:
    - `src` and `trt` (short for target) hold the batch of source and target sequences respectively.
    - The `index_to_en` function is used to convert these sequences from numerical indices back to readable text.
    - The `sample` number and corresponding source and target texts are printed out.

After printing the first 10 samples, the code continues to iterate through the dataset:

- It prints the shape of the target and source tensors for the next batch, which provides information about the number of tokens and batch size.
- The `index_to_en` function is again used to convert the first sequence of the batch from indices to text for both source and target.
- Only the first pair of the remaining batches is printed, and then the loop breaks.

This process is useful for verifying that the data loader is functioning correctly and that the sequences are being properly transformed.


In [20]:
dataset = iter(train_dataloader)
for sample in range(10):
    src, tgt = next(dataset)
    print("sample",sample)
    print("sorce:",index_to_en(src))
    print("\n")
    print("target:",index_to_en(tgt))
    print("\n")

sample 0
sorce: plot , good luck because you ' ll need it , either that or lots of coffee or soda to keep you awake ! 4/10 . . . and that


target: , good luck because you ' ll need it , either that or lots of coffee or soda to keep you awake ! 4/10 . . . and that '


sample 1
sorce: is writer/director ulli lommel . but the worst of it is that blockbusters is actually renting this to their customers ! be advised . leave this crap where it belongs


target: writer/director ulli lommel . but the worst of it is that blockbusters is actually renting this to their customers ! be advised . leave this crap where it belongs .


sample 2
sorce: . starting at about the halfway point , the beautiful and erotic arielle dombasle starts disrobing at every opportunity . that is the only thing that made this movie worth


target: starting at about the halfway point , the beautiful and erotic arielle dombasle starts disrobing at every opportunity . that is the only thing that made this movie worth watc

In [21]:
for  src,trt in dataset:
    print(trt.shape)
    print(src.shape)
    print(index_to_en(src[0,:]))
    print(index_to_en(trt[0,:]))
    break

torch.Size([30, 1])
torch.Size([30, 1])
leather
straddling


### **Masking**

In transformers, masking is crucial for ensuring certain positions are not attended to. The function ```generate_square_subsequent_mask``` produces an upper triangular matrix, which ensures that during decoding, a token can't attend to future tokens of target.


In [22]:
def get_square_subsequent_mask(seq_len, device = device):
    mask = (torch.triu(torch.ones((seq_len,seq_len),device=device)) == 1).transpose(0,1)
    mask = mask.float().masked_fill(mask == 0,float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask    

#### create a mask

In [23]:
def create_mask (src, device=device):
    src_len = src.shape[0]
    src_mask = get_square_subsequent_mask(src_len,device)
    src_padding_mask = (src == PAD_IDX).transpose(0,1)
    return src_mask, src_padding_mask

#### Testing Mask Functions 

In [24]:
src[0:4] = PAD_IDX
src[0:4]

tensor([[1],
        [1],
        [1],
        [1]])

In [25]:
mask, padding_mask = create_mask(src)
print("src", src)


src tensor([[    1],
        [    1],
        [    1],
        [    1],
        [15140],
        [    5],
        [  222],
        [   41],
        [ 5085],
        [    5],
        [ 1405],
        [    5],
        [ 3185],
        [    5],
        [ 3400],
        [    5],
        [17393],
        [    5],
        [ 1396],
        [    5],
        [  473],
        [    5],
        [  643],
        [    5],
        [11789],
        [    5],
        [20861],
        [    5],
        [  870],
        [ 2632]])


In [26]:
print("\nmask", mask)



mask tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 

In [27]:
print("\npadding mask", padding_mask)


padding mask tensor([[ True,  True,  True,  True, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False]])


### **Positional Encoding**

The Transformer model does not inherently understand the order of tokens in a sequence. To provide this information, positional encodings are added to the token embeddings, following a predefined pattern based on each token’s position.

In contrast, GPT uses learnable (trainable) positional encodings. Instead of fixed patterns like the sinusoidal encodings introduced in the original Transformer, these encodings are treated as parameters that are updated during training.

Each position in the input sequence is assigned its own learnable vector, with the same dimensionality as the token embeddings. As training progresses, the model adjusts these vectors along with its other parameters, enabling it to better capture positional relationships.

This approach allows GPT to develop more flexible and task-specific representations of position, which can enhance performance across different natural language processing tasks.

For simplicity, however, this lab uses fixed positional encodings.

In [28]:
class PositionalEncoding(nn.Module):
    
    def __init__(self, seq_len, d_model, dropout):
        super(PositionalEncoding).__init__()
        self.dropout = nn.Dropout(dropout)
        
        positions = torch.arange(0,seq_len).float().unsqueeze(1)
        
        pe = torch.zeros((seq_len,d_model))
        
        div_term = torch.exp(
            torch.arange(0,d_model,2) * (-math.log(10000)/d_model)
        )
        
        pe[:,0::2] = torch.sin(positions * div_term)
        pe[:,1::2] = torch.cos(positions * div_term)
        
        pe = pe.unsuqeeze(0)
        
        self.register_buffer("pe", pe)
        
    def forward(self, word_embedding):
        len_word_embed = word_embedding.size(0)
        
        positional_embeddings = word_embedding + self.pe[:,0:len_word_embed, :]
        
        return self.dropout(positional_embeddings)

### **Token Embedding**

Token embedding (also called word embedding or word representation) is a technique used to map words or tokens from a text corpus into numerical vectors within a continuous vector space. Each unique token is represented by a fixed-length vector, where the values capture linguistic characteristics such as meaning, context, and relationships with other words.

The TokenEmbedding class below transforms numerical token indices into their corresponding embedding vectors.

#### **When you define nn.Embedding(vocab_size, d_model), PyTorch creates a learnable embedding matrix of size:**

vocab_size×d_model
Each row corresponds to a word/token in the vocabulary
Each row is a vector of size d_model representing that token

This matrix doesn’t initially “contain meaning” — it starts with random values and learns meaningful representations during training.

What happens in forward()

When you pass seq_tokens (which are token indices):

The embedding layer looks up the corresponding rows from the embedding matrix
So for a sequence of length N, you get an output of shape:
N×d_model

That means:

Each token in the sequence is converted into its embedding vector
All these vectors together form the embedding matrix for that sequence

In [29]:
class TokenEmbedding(nn.Module):
    
    def __init__(self,vocab_size,d_model):
        self.embeddings = nn.Embedding(vocab_size,d_model)
        self.d_model = d_model
        
    def forward(self, tokens):
        embed_out = self.embeddings(tokens.long())
        return embed_out * math.sqrt(self.d_model)
        

## Custom GPT model architecture

The `CustomGPTModel` class defines a transformer-based model architecture for generative pre-trained models. This model aims to generate text and perform various NLP tasks. Below is an explanation of the main components of the class:

- **Initialization (`__init__`)**: The constructor takes several parameters including `embed_size`, `vocab_size`, `num_heads`, `num_layers`, `max_seq_len`, and `dropout`. It initializes the embedding layer, positional encoding, transformer encoder layers, and a linear layer (`lm_head`) for generating logits over the vocabulary.

- **Weight initialization (`init_weights`)**: This method initializes the weights of the model for better training convergence. The Xavier uniform initialization is used, which is a common practice for initializing weights in deep learning.

- **Decoder (`decoder`)**: Although named `decoder`, this method currently functions as the forward pass through the transformer encoder layers, followed by the generation of logits for the language modeling task. It handles the addition of positional encodings to the embeddings and applies a mask if necessary.

- **Forward pass (`forward`)**: This method is similar to the `decoder` method and defines the forward computation of the model. It processes the input through embedding layers, positional encoding, transformer encoder layers, and produces the final output using the `lm_head`.

- **Mask generation**: Both `decoder` and `forward` methods contain logic to generate a square causal mask if no source mask is provided. This mask ensures that the prediction for a position does not depend on the future tokens in the sequence, which is important for the autoregressive nature of GPT models.

- **Commented out decoder**: A section of the code is commented out, suggesting an initial design where a transformer decoder layer was considered. However, the final implementation uses only encoder layers, which is a common simplification for models focusing on language modeling and generation.

This class effectively encapsulates the necessary components to create a GPT-like model, allowing for training on language modeling tasks and text generation applications.
